###### ML469期末作業

學生： 余秉仁

專題：台股每日收盤價漲幅>5%預測模型

預測模型結論：

一、目前模型其實幾乎沒有預測能力：模型全部預測：不會漲>5%。
資料分布：Target=0有705筆；Target=1有22筆==>特徵不足。
對策：後續將再增加特徵值

二、目前只有兩個模型有抓到正樣本，所以現階段排名：
No1是LightGBM
No2是XGBoost
No3是Random Forest



## 載入函數


In [39]:
import os
import time
import requests
import numpy as np
import pandas as pd
import yfinance as yf
from pathlib import Path
from datetime import datetime
from dateutil.relativedelta import relativedelta
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import MinMaxScaler

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [40]:
# 1.基本設定
DATA_DIR = Path(r'C:\Users\pingy\ML469-Final PJ of 余秉仁\data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

stocks = {"2303.TW":"聯電", "2327.TW":"國巨", "2408.TW":"南亞科", "6239.TW":"力成", "2454.TW":"聯發科",
          "2308.TW":"台達電", "6789.TW":"采鈺", "3037.TW":"欣興", "3105.TWO":"穩懋", "3491.TWO":"昇達科"}
START_DATE = "2023-05-01"
END_DATE = "2026-06-01"


In [41]:
# 2.下載CSV
def download_stock_data():
    for ticker, name in stocks.items():
        print(f'下載{ticker}{name}')

        df = yf.download(ticker, start=START_DATE, end=END_DATE, auto_adjust=False, progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
            
        df = df.reset_index()
        df['Stock_ID'] = ticker
        df['Stock_Name'] = name

        file_path = DATA_DIR / f'{ticker}_{name}.csv'
        df.to_csv(file_path, index=False, encoding='utf-8-sig')

        print(f'已儲存:{file_path}')


In [42]:
# 3.建立特徵與目標變數
def create_features(df):
    df = df.copy()
    df.columns = df.columns.str.strip()
    df['Date'] = pd.to_datetime(df['Date'])

    numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.sort_values('Date')
    
    #每日漲幅
    df['Return'] = df['Close'].pct_change()

    #Y:隔日是否漲幅>5%
    df['Next_Return'] = (df['Close'].shift(-1) - df['Close']) / df['Close']
    df['Target'] = (df['Next_Return'] > 0.05).astype(int)
    #技術特徵
    df['MA5'] = df['Close'].rolling(5).mean()
    df['MA10'] = df['Close'].rolling(10).mean()
    df['MA20'] = df['Close'].rolling(20).mean()

    df['Volume_MA5'] = df['Volume'].rolling(5).mean()
    df['Volume_MA20'] = df['Volume'].rolling(20).mean()

    df['Range'] = (df['High'] - df['Low']) / df['Close']

    df = df.dropna()

    return df


In [50]:
# 4.評估函數
def evaluate_model(stock, model_name, y_test, pred, prod):
    try:
        auc = roc_auc_score(y_test, prob)
    except:
        auc = np.nan

    return {
        'Stock': stock,
        'Model':model_name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
        'AUC': auc
    }


In [51]:
# 5.單檔股票模型訓練
def run_models_for_stock(csv_file):
    df = pd.read_csv(csv_file)
    df2 = create_features(df)
    feature_cols = [
        'Open',
        'High',
        'Low',
        'Close',
        'Volume',
        'Return',
        'MA5',
        'MA10',
        'MA20',
        'Volume_MA5',
        'Volume_MA20',
        'Range'
    ]
    X = df2[feature_cols]
    y = df2['Target']
    # 若測試集中完全沒有Target=1, AUC會無法計算
    split_idx = int(len(df2)*0.7)
    X_train = X.iloc[:split_idx]
    X_test = X.iloc[split_idx:]
    y_train = y.iloc[:split_idx]
    y_test = y.iloc[split_idx:]

    stock_name = csv_file.stem
    results = []
    models = {
        "Random Forest": RandomForestClassifier(
            n_estimators=500,
            max_depth=6,
            class_weight="balanced",
            random_state=42
        ),
        "XGBoost": XGBClassifier(
            n_estimators=500,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=4,
            class_weight="balanced",
            random_state=42,
            verbose=-1
        )
    }

    for model_name, model in models.items():

        model.fit(X_train, y_train)

        pred = model.predict(X_test)
        prob = model.predict_proba(X_test)[:, 1]

        results.append(
            evaluate_model(
                stock_name,
                model_name,
                y_test,
                pred,
                prob
            )
        )

    return pd.DataFrame(results)

In [52]:
# 6.主程式
import traceback
download_stock_data()

all_results = []

csv_files = [f for f in DATA_DIR.glob("*.csv") if "Model_Result" not in f.name and "model_performance" not in f.name]
for csv_file in csv_files:
    print(f'\n開始訓練:{csv_file.name}')
    
    try:
        result_df = run_models_for_stock(csv_file)
        all_results.append(result_df)
        print('完成')
         
    except Exception as e:
        print(f"{csv_file.name} 失敗：{e}")
        traceback.print_exc()


下載2303.TW聯電
已儲存:C:\Users\pingy\ML469-Final PJ of 余秉仁\data\2303.TW_聯電.csv
下載2327.TW國巨
已儲存:C:\Users\pingy\ML469-Final PJ of 余秉仁\data\2327.TW_國巨.csv
下載2408.TW南亞科
已儲存:C:\Users\pingy\ML469-Final PJ of 余秉仁\data\2408.TW_南亞科.csv
下載6239.TW力成
已儲存:C:\Users\pingy\ML469-Final PJ of 余秉仁\data\6239.TW_力成.csv
下載2454.TW聯發科
已儲存:C:\Users\pingy\ML469-Final PJ of 余秉仁\data\2454.TW_聯發科.csv
下載2308.TW台達電
已儲存:C:\Users\pingy\ML469-Final PJ of 余秉仁\data\2308.TW_台達電.csv
下載6789.TW采鈺
已儲存:C:\Users\pingy\ML469-Final PJ of 余秉仁\data\6789.TW_采鈺.csv
下載3037.TW欣興
已儲存:C:\Users\pingy\ML469-Final PJ of 余秉仁\data\3037.TW_欣興.csv
下載3105.TWO穩懋
已儲存:C:\Users\pingy\ML469-Final PJ of 余秉仁\data\3105.TWO_穩懋.csv
下載3491.TWO昇達科
已儲存:C:\Users\pingy\ML469-Final PJ of 余秉仁\data\3491.TWO_昇達科.csv

開始訓練:2303.TW_聯電.csv
完成

開始訓練:2308.TW_台達電.csv
完成

開始訓練:2327.TW_國巨.csv
完成

開始訓練:2408.TW_南亞科.csv
完成

開始訓練:2454.TW_聯發科.csv
完成

開始訓練:3037.TW_欣興.csv
完成

開始訓練:3105.TWO_穩懋.csv
完成

開始訓練:3491.TWO_昇達科.csv
完成

開始訓練:6239.TW_力成.csv
完成

開始訓練:6789.TW_采鈺.csv
完成


In [53]:
# 7.彙整與模型排名
if len(all_results) == 0:

    print("沒有任何模型成功完成訓練，請檢查前面錯誤訊息。")

else:

    final_result = pd.concat(
        all_results,
        ignore_index=True
    )

    final_result = final_result.sort_values(
        by=["AUC", "F1", "Recall"],
        ascending=False
    )

    final_result["Rank"] = range(
        1,
        len(final_result) + 1
    )

    output_file = DATA_DIR / "model_performance_summary.csv"

    final_result.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("全部模型訓練完成")
    print(final_result)
    print(f"績效彙整已儲存：{output_file}")

全部模型訓練完成
           Stock          Model  Accuracy  Precision    Recall        F1  AUC  \
14   2454.TW_聯發科       LightGBM  0.899543   1.000000  0.043478  0.083333  NaN   
11   2408.TW_南亞科       LightGBM  0.730594   0.076923  0.020833  0.032787  NaN   
0     2303.TW_聯電  Random Forest  0.913242   0.000000  0.000000  0.000000  NaN   
1     2303.TW_聯電        XGBoost  0.913242   0.000000  0.000000  0.000000  NaN   
2     2303.TW_聯電       LightGBM  0.913242   0.000000  0.000000  0.000000  NaN   
3    2308.TW_台達電  Random Forest  0.858447   0.000000  0.000000  0.000000  NaN   
4    2308.TW_台達電        XGBoost  0.858447   0.000000  0.000000  0.000000  NaN   
5    2308.TW_台達電       LightGBM  0.858447   0.000000  0.000000  0.000000  NaN   
6     2327.TW_國巨  Random Forest  0.876712   0.000000  0.000000  0.000000  NaN   
7     2327.TW_國巨        XGBoost  0.876712   0.000000  0.000000  0.000000  NaN   
8     2327.TW_國巨       LightGBM  0.876712   0.000000  0.000000  0.000000  NaN   
9    2408.TW_南亞科  R

In [36]:
print(type(all_results))
print(len(all_results))

<class 'list'>
0
